In [1]:
import nest_asyncio
nest_asyncio.apply()

import os
import glob
import pandas as pd
import seaborn as sns
import dataframe_image as dfi
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

In [57]:
# --- Configuration ---

# The path to the folder the your CSV files.
FOLDER_PATH = '/fast/AG_Kainmueller/vguarin/aggrigator_experiments/output/tables/auroc_gmm/'

# This dictionary maps the aggregator names in the CSV files
AGGREGATOR_NAME_MAPPING = {
    'Mean': 'AVG',
    'Quantile 0.6': 'AQA 0.60',
    'Quantile 0.75': 'AQA 0.75',
    'Quantile 0.9': 'AQA 0.90',
    'Patch 10': 'PLM 10',
    'Patch 20': 'PLM 20',
    'Patch 50': 'PLM 50',
    'Threshold 0.3': 'ATA 0.3',
    'Threshold 0.5': 'ATA 0.5',
    'Threshold 0.7': 'ATA 0.7',
    'Quantile fg. ratio': 'QFR',
    'Imbalance-w. class avg.': 'ICA', # Assuming this mapping
    'Equally-w. class avg.': 'BCA', # Assuming this mapping
    'GMM_pixel': 'GMM-Int',           # Assuming this mapping
    'GMM_spatial': 'GMM-Spa',         # Assuming this mapping
    'GMM': 'GMM-All',                 # Assuming this mapping
}

COLUMN_NAME_MAPPING = {
    'instance_lizard_glas_set_pu': 'LIZ-IG',
    'fgbg_wormbodies_protists_pu': 'WORM-Pro',
    'instance_arctique_nuclei_intensity_pu': 'ARC-Nuc',
    'semantic_gta_cityscapes_pu': 'CAR-CS',
    'crops_vs_weed_weedsgalore_maize_pu': 'WEED-Hand',
    'fgbg_wormbodies_nematodes_pu': 'WORM-Nem',
    'fgbg_lidc_malignancy_pu': 'LIDC-Mal',
    'semantic_lizard_glas_set_pu': 'LIZ-SG',
    'fgbg_lidc_texture_pu': 'LIDC-Tex',
    'semantic_arctique_blood_cells_pu': 'ARC-BC'
}

# --- Data Loading and Processing ---

# Find all relevant CSV files in the specified folder
search_pattern = os.path.join(FOLDER_PATH, '*_auroc_ood_results.csv')
file_paths = glob.glob(search_pattern)

if not file_paths:
    print(f"Error: No CSV files found at '{search_pattern}'. Please check your FOLDER_PATH.")
else:
    print(f"Found {len(file_paths)} CSV files to process.")

all_means = []
all_stds = []

for filepath in file_paths:
    # Extract the dataset name from the filename.
    basename = os.path.basename(filepath)
    dataset_name = basename.replace('_auroc_ood_results.csv', '')

    # Read AUROC and AUROC_std ---
    temp_df = pd.read_csv(filepath)
    temp_df['Aggregator'] = temp_df['Aggregator'].map(AGGREGATOR_NAME_MAPPING).fillna(temp_df['Aggregator'])
    temp_df = temp_df.set_index('Aggregator')

    # Create and append the means DataFrame
    mean_df = temp_df[['AUROC']].rename(columns={'AUROC': dataset_name})
    all_means.append(mean_df)
    
    std_df = temp_df[['AUROC_std']].rename(columns={'AUROC_std': dataset_name})
    all_stds.append(std_df)
    
# Numeric DataFrame for calculations and color mapping
summary_df = pd.concat(all_means, axis=1)
stds_df = pd.concat(all_stds, axis=1)

summary_df = summary_df.rename(columns=COLUMN_NAME_MAPPING)
stds_df = stds_df.rename(columns=COLUMN_NAME_MAPPING)

# --- Calculate Rank, Reorder, and Sort ---
ranks_df = summary_df.rank(ascending=False, method='min')
summary_df['Mean Rank'] = ranks_df.mean(axis=1)
final_column_order = [
    'ARC-BC', 'ARC-Nuc', 'CAR-CS', 'LIDC-Mal', 'LIDC-Tex',
    'LIZ-IG', 'LIZ-SG', 'WEED-Hand', 'WORM-Nem', 'WORM-Pro', 'Mean Rank'
]
summary_df = summary_df[final_column_order]
summary_df = summary_df.sort_values(by='Mean Rank', ascending=True)

# Align stds_df to the final sorted summary_df
stds_df = stds_df.reindex(index=summary_df.index, columns=summary_df.columns.drop('Mean Rank', errors='ignore'))
summary_df.index.name = None

# Create a second DataFrame with the desired string formats for display
display_df = pd.DataFrame(index=summary_df.index, columns=summary_df.columns, dtype=str)
for col in display_df.columns:
    if col == 'Mean Rank':
        display_df[col] = summary_df[col].map('{:.1f}'.format)
    else:
        # Create "mean ± std" strings
        mean_series = summary_df[col]
        std_series = stds_df[col]
        display_df[col] = mean_series.map('{:.3f}'.format) + ' ± ' + std_series.map('{:.3f}'.format)

# --- Final Styling ---
dataset_cols = [col for col in summary_df.columns if col != 'Mean Rank']

# Apply heatmap styling similar to the example image
# Green for high values, red for low, white for middle
styled_df = display_df.style.background_gradient(
    cmap=sns.diverging_palette(10, 130, as_cmap=True), # Red to Green palette #'RdYlGn',
    gmap=summary_df[dataset_cols],
    axis=None, 
    low=0.3, # Adjust these to control the color intensity
    high=0.7
).set_properties(
    **{'width': '100px'}
)

print("\n--- Styled Summary Table with Mean Rank ---")
# Display the styled DataFrame in the notebook
display(styled_df)

# --- Save the DataFrame for Sharing ---

# You can save the data in several formats.

# a) Save the raw data (without styles) to a CSV file
output_csv_path = 'auroc_summary_with_ranks.csv'
summary_df.to_csv(output_csv_path)
print(f"\nSuccessfully saved data to '{output_csv_path}'")

# b) Export using the Matplotlib backend
output_image_path = 'auroc_summary_table.png'

dfi.export(
    styled_df,
    output_image_path,
    table_conversion='matplotlib' # Ensures we use the reliable backend
)

print(f"Successfully saved styled table as a PNG to '{output_image_path}' using the Matplotlib backend.")

Found 10 CSV files to process.

--- Styled Summary Table with Mean Rank ---


,ARC-BC,ARC-Nuc,CAR-CS,LIDC-Mal,LIDC-Tex,LIZ-IG,LIZ-SG,WEED-Hand,WORM-Nem,WORM-Pro,Mean Rank
BCA,0.715 ± 0.063,0.787 ± 0.054,0.893 ± 0.012,0.568 ± 0.048,0.816 ± 0.072,0.677 ± 0.024,0.681 ± 0.024,0.582 ± 0.065,0.769 ± 0.058,0.945 ± 0.023,5.2
GMM-All,0.836 ± 0.051,0.868 ± 0.047,0.996 ± 0.004,0.861 ± 0.034,0.766 ± 0.054,0.451 ± 0.027,0.437 ± 0.030,0.955 ± 0.028,0.997 ± 0.003,1.000 ± 0.000,5.2
ICA,0.602 ± 0.073,0.829 ± 0.048,0.845 ± 0.017,0.566 ± 0.050,0.815 ± 0.072,0.713 ± 0.024,0.588 ± 0.028,0.574 ± 0.057,0.766 ± 0.058,0.945 ± 0.021,5.9
GMM-Int,0.787 ± 0.055,0.826 ± 0.058,0.727 ± 0.020,0.863 ± 0.035,0.779 ± 0.049,0.487 ± 0.026,0.433 ± 0.026,0.911 ± 0.041,0.984 ± 0.011,1.000 ± 0.000,6.0
QFR,0.623 ± 0.068,0.864 ± 0.045,0.625 ± 0.023,0.542 ± 0.048,0.886 ± 0.053,0.678 ± 0.025,0.573 ± 0.025,0.579 ± 0.063,0.675 ± 0.064,0.911 ± 0.031,6.6
GMM-Spa,0.931 ± 0.032,0.663 ± 0.071,1.000 ± 0.000,0.670 ± 0.044,0.720 ± 0.075,0.494 ± 0.028,0.415 ± 0.026,0.850 ± 0.046,0.894 ± 0.039,0.899 ± 0.030,7.1
PLM 50,0.476 ± 0.076,0.683 ± 0.067,0.457 ± 0.024,0.955 ± 0.019,0.502 ± 0.074,0.680 ± 0.025,0.749 ± 0.023,0.566 ± 0.046,0.486 ± 0.068,0.859 ± 0.036,8.7
AQA 0.60,0.534 ± 0.072,0.638 ± 0.072,0.636 ± 0.023,0.954 ± 0.020,0.505 ± 0.078,0.762 ± 0.024,0.806 ± 0.021,0.331 ± 0.053,0.487 ± 0.068,0.549 ± 0.055,8.9
PLM 10,0.524 ± 0.078,0.640 ± 0.069,0.436 ± 0.025,0.907 ± 0.027,0.590 ± 0.090,0.651 ± 0.024,0.653 ± 0.026,0.492 ± 0.041,0.691 ± 0.063,0.857 ± 0.040,9.1
PLM 20,0.472 ± 0.075,0.678 ± 0.067,0.417 ± 0.026,0.955 ± 0.019,0.516 ± 0.081,0.673 ± 0.025,0.727 ± 0.023,0.562 ± 0.044,0.567 ± 0.069,0.838 ± 0.042,9.2



Successfully saved data to 'auroc_summary_with_ranks.csv'
Successfully saved styled table as a PNG to 'auroc_summary_table.png' using the Matplotlib backend.


In [55]:
# --- Configuration ---
FOLDER_PATH = '/fast/AG_Kainmueller/vguarin/aggrigator_experiments/output/tables/eaurc_id_ood/'
AGGREGATOR_NAME_MAPPING = {
    'Mean': 'AVG', 'Quantile 0.6': 'AQA 0.60', 'Quantile 0.75': 'AQA 0.75',
    'Quantile 0.9': 'AQA 0.90', 'Patch 10': 'PLM 10', 'Patch 20': 'PLM 20',
    'Patch 50': 'PLM 50', 'Threshold 0.3': 'ATA 0.3', 'Threshold 0.5': 'ATA 0.5',
    'Threshold 0.7': 'ATA 0.7', 'Quantile fg. ratio': 'QFR',
    'Imbalance-w. class avg.': 'ICA', 'Equally-w. class avg.': 'BCA',
    'GMM_pixel': 'GMM-I', 'GMM_spatial': 'GMM-S', 'GMM': 'GMM-F',
}
COLUMN_NAME_MAPPING = {
    'instance_lizard_glas_set_pu': 'LIZ-IG', 'fgbg_wormbodies_protists_pu': 'WORM-Pro',
    'instance_arctique_nuclei_intensity_pu': 'ARC-Nuc', 'semantic_gta_cityscapes_pu': 'CAR-CS',
    'crops_vs_weed_weedsgalore_maize_pu': 'WEED-Hand', 'fgbg_wormbodies_nematodes_pu': 'WORM-Nem',
    'fgbg_lidc_malignancy_pu': 'LIDC-Mal', 'semantic_lizard_glas_set_pu': 'LIZ-SG',
    'fgbg_lidc_texture_pu': 'LIDC-Tex', 'semantic_arctique_blood_cells_pu': 'ARC-BC'
}

# --- Data Loading and Processing ---
search_pattern = os.path.join(FOLDER_PATH, '*_eaurc_id_ood_results.csv')
file_paths = glob.glob(search_pattern)
all_means, all_stds = [], []
for filepath in file_paths:
    basename = os.path.basename(filepath)
    dataset_name = basename.replace('_eaurc_id_ood_results.csv', '')
    temp_df = pd.read_csv(filepath)
    temp_df['Aggregator'] = temp_df['Aggregator'].map(AGGREGATOR_NAME_MAPPING).fillna(temp_df['Aggregator'])
    temp_df = temp_df.set_index('Aggregator')
    all_means.append(temp_df[['EAURC']].rename(columns={'EAURC': dataset_name}))
    all_stds.append(temp_df[['EAURC_std']].rename(columns={'EAURC_std': dataset_name}))

summary_df_eaurc = pd.concat(all_means, axis=1)
stds_df_eaurc = pd.concat(all_stds, axis=1)
summary_df_eaurc = summary_df_eaurc.rename(columns=COLUMN_NAME_MAPPING)
stds_df_eaurc = stds_df_eaurc.rename(columns=COLUMN_NAME_MAPPING)

# --- Calculate Mean Rank (where LOWEST is better) ---
ranks_df_eaurc = summary_df_eaurc.rank(ascending=True, method='min')
summary_df_eaurc['Mean Rank'] = ranks_df_eaurc.mean(axis=1)
final_column_order = [
    'ARC-BC', 'ARC-Nuc', 'CAR-CS', 'LIDC-Mal', 'LIDC-Tex',
    'LIZ-IG', 'LIZ-SG', 'WEED-Hand', 'WORM-Nem', 'WORM-Pro', 'Mean Rank'
]
summary_df_eaurc = summary_df_eaurc[final_column_order]
summary_df_eaurc = summary_df_eaurc.sort_values(by='Mean Rank', ascending=True)
stds_df_eaurc = stds_df_eaurc.reindex(index=summary_df_eaurc.index, columns=summary_df_eaurc.columns.drop('Mean Rank', errors='ignore'))
summary_df_eaurc.index.name = None

# --- Create the Display DataFrame ---
display_df_eaurc = pd.DataFrame(index=summary_df_eaurc.index, columns=summary_df_eaurc.columns, dtype=str)
for col in display_df_eaurc.columns:
    if col == 'Mean Rank':
        # This formats the rank to one decimal place
        display_df_eaurc[col] = summary_df_eaurc[col].map('{:.1f}'.format)
    else:
        mean_series = summary_df_eaurc[col]
        std_series = stds_df_eaurc[col]
        display_df_eaurc[col] = mean_series.map('{:.3f}'.format) + ' ± ' + std_series.map('{:.3f}'.format)

# --- Final Styling ---
dataset_cols_eaurc = [col for col in summary_df_eaurc.columns if col != 'Mean Rank']

# --- THIS IS THE CORRECTED STYLING BLOCK ---
# Start with the base styler object from the correct display DataFrame
styled_df_eaurc = display_df_eaurc.style

# Loop through each data column and apply the gradient individually
for col in dataset_cols_eaurc:
    styled_df_eaurc = styled_df_eaurc.background_gradient(
        cmap=sns.diverging_palette(10, 130, as_cmap=True).reversed(),  # Reversed cmap for "lower is better"
        subset=[col],      # Apply to this specific column
        gmap=summary_df_eaurc[col], # Use the corresponding numeric column for color
        low=0.3,
        high=0.7
    )

# Chain the final properties after the loop
styled_df_eaurc = styled_df_eaurc.set_properties(**{'width': '100px'})


print("\n--- Styled EAURC Summary Table (Lowest is Better, Column-Normalized) ---")
display(styled_df_eaurc)


# --- Save final files ---
output_csv_path_eaurc = 'eaurc_summary_with_ranks.csv'
summary_df_eaurc.to_csv(output_csv_path_eaurc)
print(f"\nSuccessfully saved data to '{output_csv_path_eaurc}'")

# Export the final styled image
output_image_path_eaurc = 'eaurc_summary_table.png'
dfi.export(
    styled_df_eaurc,
    output_image_path_eaurc,
    table_conversion='matplotlib'
)
print(f"Successfully saved styled table as a PNG to '{output_image_path_eaurc}'.")


--- Styled EAURC Summary Table (Lowest is Better, Column-Normalized) ---


,ARC-BC,ARC-Nuc,CAR-CS,LIDC-Mal,LIDC-Tex,LIZ-IG,LIZ-SG,WEED-Hand,WORM-Nem,WORM-Pro,Mean Rank
QFR,0.039 ± 0.008,0.017 ± 0.003,0.062 ± 0.006,0.049 ± 0.006,0.087 ± 0.040,0.256 ± 0.012,0.269 ± 0.012,0.157 ± 0.018,0.038 ± 0.009,0.038 ± 0.006,3.4
GMM-F,0.054 ± 0.010,0.028 ± 0.004,0.054 ± 0.006,0.074 ± 0.011,0.070 ± 0.010,0.161 ± 0.013,0.207 ± 0.012,0.079 ± 0.013,0.094 ± 0.012,0.058 ± 0.008,4.2
GMM-I,0.065 ± 0.012,0.029 ± 0.005,0.095 ± 0.009,0.071 ± 0.009,0.064 ± 0.008,0.173 ± 0.013,0.202 ± 0.012,0.087 ± 0.016,0.087 ± 0.010,0.060 ± 0.008,5.4
GMM-S,0.046 ± 0.008,0.040 ± 0.006,0.046 ± 0.004,0.075 ± 0.011,0.076 ± 0.010,0.176 ± 0.012,0.202 ± 0.014,0.122 ± 0.018,0.133 ± 0.033,0.069 ± 0.009,6.0
BCA,0.036 ± 0.007,0.028 ± 0.004,0.044 ± 0.006,0.065 ± 0.012,0.129 ± 0.041,0.296 ± 0.012,0.334 ± 0.011,0.201 ± 0.020,0.099 ± 0.027,0.092 ± 0.011,6.7
ATA 0.3,0.053 ± 0.012,0.031 ± 0.005,0.086 ± 0.008,0.122 ± 0.031,0.139 ± 0.041,0.268 ± 0.013,0.281 ± 0.012,0.285 ± 0.019,0.030 ± 0.008,0.088 ± 0.012,8.0
ATA 0.5,0.041 ± 0.009,0.024 ± 0.003,0.102 ± 0.009,0.186 ± 0.037,0.147 ± 0.041,0.233 ± 0.013,0.284 ± 0.012,0.287 ± 0.018,0.027 ± 0.007,0.083 ± 0.012,8.1
PLM 10,0.064 ± 0.013,0.026 ± 0.004,0.087 ± 0.008,0.088 ± 0.030,0.132 ± 0.042,0.270 ± 0.013,0.305 ± 0.012,0.297 ± 0.022,0.113 ± 0.037,0.054 ± 0.008,8.3
PLM 20,0.062 ± 0.011,0.025 ± 0.005,0.089 ± 0.009,0.097 ± 0.027,0.141 ± 0.042,0.279 ± 0.013,0.328 ± 0.011,0.288 ± 0.021,0.095 ± 0.031,0.063 ± 0.009,8.5
ATA 0.7,0.126 ± 0.020,0.015 ± 0.003,0.102 ± 0.009,0.203 ± 0.037,0.151 ± 0.041,0.215 ± 0.013,0.254 ± 0.013,0.302 ± 0.022,0.035 ± 0.009,0.077 ± 0.011,9.4



Successfully saved data to 'eaurc_summary_with_ranks.csv'
Successfully saved styled table as a PNG to 'eaurc_summary_table.png'.
